Example of Notebook that uses Pytorch Lightning and DeepSpeed.   DeepSpeed allows for training that spans multiple machines with multiple gpus per machine.  It lets traning scale!

Install Lightning and DeepSpeed

In [0]:
%pip install pytorch-lightning deepspeed transformers datasets
dbutils.library.restartPython()

Example code for defining a LightningModule

Create a DeepSpeed ZeRO-2 Config.
Tune it for H100 memory + bandwidth

Use ZeRO-3 + CPU offload config if we are going to train on something that is WAY BIGGER than fits in H100 memory.

train_batch_size should be scaled global effective batch size (in a multi-node, multi-gpu setup)

Formula for Effective (Global) Batch Size

DeepSpeed’s "train_batch_size" should equal:
```global_batch_size = per_gpu_batch_size × gradient_accumulation_steps × world_size``

where:
	•	per_gpu_batch_size = how many samples each GPU processes in one forward/backward step
	•	gradient_accumulation_steps = how many mini-batches you accumulate before doing an optimizer step
	•	world_size = total number of GPUs = (num_nodes × gpus_per_node)

This Case:
	• num_nodes = 4
	•	gpus_per_node = 4 (g5.12xlarge has 4× A10G GPUs)
	•	→ world_size = 4 × 4 = 16


safe for A10G memory - 24 GB:
	•	per_gpu_batch_size = 8
	•	gradient_accumulation_steps = 2
	•	global_batch_size = 8 × 2 × 16 = 256

In [0]:
import json
# ZeRO-2
ds_config = {
    "train_batch_size": 256,
    "gradient_accumulation_steps": 2,
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "reduce_scatter": True,
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "reduce_bucket_size": 2e8
    },
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 0.00015,
            "betas": [0.9, 0.95],
            "eps": 1e-8,
            "weight_decay": 0.01
        }
    },
    "scheduler": {
        "type": "WarmupLR",
        "params": {
            "warmup_min_lr": 0,
            "warmup_max_lr": 0.00015,
            "warmup_num_steps": 1000
        }
    }
}

with open("/local_disk0/ds_config.json", "w") as f:
    json.dump(ds_config, f, indent=2)

print(open("/local_disk0/ds_config.json").read())

'''
ZeRO-3 Config - For large models - offloads to CPU

ds_config = {
    "train_batch_size": 1024,
    "gradient_accumulation_steps": 2,
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "none"},   # keep params on GPU, offload optimizer only
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": 2e8,
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9
    },
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 0.00015,
            "betas": [0.9, 0.95],
            "eps": 1e-8,
            "weight_decay": 0.01
        }
    },
    "scheduler": {
        "type": "WarmupLR",
        "params": {
            "warmup_min_lr": 0,
            "warmup_max_lr": 0.00015,
            "warmup_num_steps": 1000
        }
    }
}
'''

Launch Training

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

def entrypoint():
    # Import and call your script’s main()
    import DeepSpeedTrain
    DeepSpeedTrain.main()

# GPUs per node on g5.12xlarge = 4
TorchDistributor(num_processes=4, local_mode=False, use_gpu=True).run(entrypoint)

In [0]:
%env NCCL_DEBUG=INFO
!python DeepSpeedTrain.py